# vector_checker — разбор расхождений датасет ↔ Excel 1С

Универсальный tool: в ячейке конфига задай пути к **запросу 1С** и **датасету** любой модели.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vector_checker.compare_features import (
    compare_feature_frames,
    load_excel_export,
    mismatch_feature_counts,
    report_to_frame,
    side_by_side_for_loss,
)
from vector_checker.paths import EXCEL_DIR, KEY_COL, ensure_work_dirs
from vector_checker.query_inject import sample_loss_numbers
from vector_checker.synthetic import write_synthetic_fixtures

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
ensure_work_dirs()
print("REPO_ROOT:", REPO_ROOT)

## 1. Конфиг путей

- `USE_SYNTHETIC=True` — локальный smoke без 1С
- иначе укажи `QUERY_PATH`, `DF_PATH`, `EXCEL_NAME`

In [ ]:
USE_SYNTHETIC = True
SYNTHETIC_MISMATCH = True

# --- боевой режим ---
QUERY_PATH = REPO_ROOT / "queries" / "Сутяжность.txt"
DF_PATH = Path(r"C:\path\to\df_for_service.parquet")  # замени
EXCEL_NAME = "export.xlsx"  # файл в work/excel/

N_LOSSES: int | None = None
RANDOM_SEED = 42
KEY = KEY_COL  # обычно LOSS_NUMBER

if USE_SYNTHETIC:
    paths = write_synthetic_fixtures()
    df_path = paths["df"]
    excel_path = paths["excel_bad" if SYNTHETIC_MISMATCH else "excel_ok"]
    query_path = paths["query_stub"]
else:
    df_path = Path(DF_PATH)
    query_path = Path(QUERY_PATH)
    excel_path = EXCEL_DIR / EXCEL_NAME
    if not df_path.is_file():
        raise FileNotFoundError(df_path)
    if not query_path.is_file():
        raise FileNotFoundError(query_path)
    if not excel_path.is_file():
        raise FileNotFoundError(f"Нет Excel: {excel_path}")

print("query:", query_path)
print("df   :", df_path)
print("excel:", excel_path)
print("sample:", N_LOSSES, "seed:", RANDOM_SEED)

In [ ]:
service_full = pd.read_parquet(df_path)
excel = load_excel_export(excel_path)

sampled = sample_loss_numbers(
    service_full[KEY].tolist(), N_LOSSES, random_seed=RANDOM_SEED
)
_keys = set(sampled)
service = service_full.loc[
    service_full[KEY].map(lambda x: str(x).strip() if pd.notna(x) else "").isin(_keys)
].copy()
if KEY in excel.columns:
    excel = excel.loc[
        excel[KEY].map(lambda x: str(x).strip() if pd.notna(x) else "").isin(_keys)
    ].copy()

print("service:", service.shape, f"(из {len(service_full)})")
print("excel:", excel.shape)
display(service.head(3))
display(excel.head(3))

## 2. Сводка и расхождения

In [ ]:
report, mismatches = compare_feature_frames(service, excel, all_overlap=True, key_col=KEY)

display(report_to_frame(report))
print("ok:", report.ok)
display(mismatch_feature_counts(mismatches))
display(mismatches)

## 3. Ключи только в одной стороне

In [ ]:
only_service = pd.DataFrame({KEY: report.only_service_losses})
only_excel = pd.DataFrame({KEY: report.only_excel_losses})
print("только в service:", len(only_service))
display(only_service.head(20))
print("только в excel:", len(only_excel))
display(only_excel.head(20))

## 4. Drill-down по одному ключу

In [ ]:
if not mismatches.empty:
    KEY_TO_INSPECT = mismatches[KEY].iloc[0]
elif report.only_service_losses:
    KEY_TO_INSPECT = report.only_service_losses[0]
else:
    KEY_TO_INSPECT = str(service[KEY].iloc[0])

print("KEY_TO_INSPECT:", KEY_TO_INSPECT)
sbs = side_by_side_for_loss(service, excel, KEY_TO_INSPECT, key_col=KEY)
display(sbs.loc[~sbs["match"]] if (~sbs["match"]).any() else sbs)
display(sbs)